# Questão 4 — Produção de vigas (PL com Gurobi)

**Objetivo.** Minimizar o custo semanal de operação de três máquinas (A, B, C) alocando horas de produção para quatro tipos de viga (pequena, mediana, grande, extra).

**Variáveis.** $h_{t,m} \ge 0$: horas da máquina $m \in \{A,B,C\}$ dedicadas ao tipo $t \in \{\text{p},\text{m},\text{g},\text{e}\}$.

**Parâmetros.**
- Taxa (m/h): $r_{t,m}$ (tabela do enunciado).
- Custo por hora: $c_A=30, c_B=50, c_C=80$.
- Limite de horas por máquina: $H_m=50$.
- Demanda (m): $d_{\text{p}}=12000,\ d_{\text{m}}=6000,\ d_{\text{g}}=5000,\ d_{\text{e}}=7000$.

**Modelo.**
$$
\min \sum_{m} c_m \sum_{t} h_{t,m}
\quad
\text{s.a.}\quad
\sum_m r_{t,m} h_{t,m} \ge d_t\ \ (\forall t),
\quad
\sum_t h_{t,m} \le H_m\ \ (\forall m),
\quad
h_{t,m}\ge 0.
$$

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gurobipy as gp
from gurobipy import Model, GRB, quicksum

In [2]:
# -----------------------------
# Dados do problema
# -----------------------------
T = ["p", "m", "g", "e"]              # tipos: pequena, mediana, grande, extra
M = ["A", "B", "C"]                   # máquinas

# taxas r_{t,m} (metros por hora) conforme a tabela
r = {
    "p": {"A": 350, "B": 650, "C": 850},
    "m": {"A": 250, "B": 400, "C": 700},
    "g": {"A": 200, "B": 350, "C": 600},
    "e": {"A": 125, "B": 200, "C": 325},
}

# demandas (metros)
d = {"p": 12000, "m": 6000, "g": 5000, "e": 7000}

# custo por hora por máquina
c = {"A": 30, "B": 50, "C": 80}

# horas disponíveis por máquina
H = {"A": 50, "B": 50, "C": 50}

# -----------------------------
# Modelo
# -----------------------------
m = Model("steel_beams")
m.Params.OutputFlag = 1  # 1 para log, 0 para silêncio

# variáveis: horas dedicadas a cada (tipo, máquina)
h = m.addVars(T, M, name="h", lb=0.0, vtype=GRB.CONTINUOUS)

# objetivo: minimizar custo total por hora
m.setObjective(quicksum(c[j] * quicksum(h[t, j] for t in T) for j in M), GRB.MINIMIZE)

# demanda por tipo (produção >= demanda)
dem_ctrs = {}
for t in T:
    dem_ctrs[t] = m.addConstr(quicksum(r[t][j] * h[t, j] for j in M) >= d[t], name=f"dem_{t}")

# limite de horas por máquina
cap_ctrs = {}
for j in M:
    cap_ctrs[j] = m.addConstr(quicksum(h[t, j] for t in T) <= H[j], name=f"cap_{j}")

# opcional: gravar o LP
#m.write("steel_beams.lp")

# otimiza
m.optimize()

# -----------------------------
# Relatórios / Saída formatada
# -----------------------------
if m.SolCount > 0:
    # Tabela de horas por (tipo, máquina)
    hours_data = {(t,j): h[t,j].X for t in T for j in M}
    hours_df = pd.Series(hours_data).unstack().reindex(index=T, columns=M)
    hours_df.index = ["pequena (p)","mediana (m)","grande (g)","extra (e)"]
    hours_df.columns = [f"Maq {j}" for j in M]

    # Produção por tipo (soma nas máquinas) e por (tipo, máquina)
    prod_data = {(t,j): r[t][j] * h[t,j].X for t in T for j in M}
    prod_df = pd.Series(prod_data).unstack().reindex(index=T, columns=M)
    prod_df.index = hours_df.index
    prod_df.columns = [f"Maq {j}" for j in M]
    prod_df["Total (m)"] = prod_df.sum(axis=1)
    prod_req = pd.Series({t: d[t] for t in T}, index=T)
    prod_df["Demanda (m)"] = prod_req.values
    prod_df["Excesso (m)"] = prod_df["Total (m)"] - prod_df["Demanda (m)"]

    # Uso total de horas por máquina
    use_hours = pd.Series({j: sum(h[t,j].X for t in T) for j in M})
    use_hours.index = [f"Maq {j}" for j in M]
    cap_hours = pd.Series({f"Maq {j}": H[j] for j in M})
    slack_hours = cap_hours - use_hours

    # Objetivo
    total_cost = m.ObjVal

    print("\n========== OBJETIVO ==========")
    print(f"Custo mínimo (US$): {total_cost:,.2f}")

    print("\n========== HORAS POR (TIPO, MÁQUINA) ==========")
    display(hours_df.style.format("{:.4f}"))

    print("\n========== PRODUÇÃO (m) POR (TIPO, MÁQUINA) ==========")
    display(prod_df.style.format("{:,.2f}"))

    print("\n========== USO DE HORAS POR MÁQUINA ==========")
    use_tbl = pd.DataFrame({
        "Uso (h)": use_hours,
        "Capacidade (h)": cap_hours,
        "Folga (h)": slack_hours
    })
    display(use_tbl.style.format("{:.4f}"))

    # (Opcional) preços sombra e custos reduzidos
    try:
        dual_demand = {t: dem_ctrs[t].Pi for t in T}
        dual_cap    = {j: cap_ctrs[j].Pi for j in M}
        print("\n========== DUAIS (preços-sombra) ==========")
        print("Demanda por tipo (US$/m):", {k: f"{v:.6f}" for k,v in dual_demand.items()})
        print("Capacidade por máquina (US$/h):", {k: f"{v:.6f}" for k,v in dual_cap.items()})
    except AttributeError:
        pass
else:
    print("Modelo inviável ou sem solução encontrada.")


Set parameter Username
Set parameter LicenseID to value 2673838
Academic license - for non-commercial use only - expires 2026-05-31
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i9-14900HX, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 7 rows, 12 columns and 24 nonzeros
Model fingerprint: 0x6d5d8b0b
Coefficient statistics:
  Matrix range     [1e+00, 9e+02]
  Objective range  [3e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 1e+04]
Presolve time: 0.01s
Presolved: 7 rows, 12 columns, 24 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.156250e+03   0.000000e+00      0s
       5    3.9600733e+03   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.02 seconds (0.00 work units)
Optimal objective  3.960073260e+0

,Maq A,Maq B,Maq C
pequena (p),0.0000,18.4615,0.0000
mediana (m),0.0000,0.0000,8.5714
grande (g),0.0000,0.0000,8.3333
extra (e),50.0000,0.0000,2.3077



========== PRODUÇÃO (m) POR (TIPO, MÁQUINA) ==========


,Maq A,Maq B,Maq C,Total (m),Demanda (m),Excesso (m)
pequena (p),0.00,"12,000.00",0.00,"12,000.00","12,000.00",-0.00
mediana (m),0.00,0.00,"6,000.00","6,000.00","6,000.00",0.00
grande (g),0.00,0.00,"5,000.00","5,000.00","5,000.00",0.00
extra (e),"6,250.00",0.00,750.00,"7,000.00","7,000.00",0.00



========== USO DE HORAS POR MÁQUINA ==========


,Uso (h),Capacidade (h),Folga (h)
Maq A,50.0000,50.0000,0.0000
Maq B,18.4615,50.0000,31.5385
Maq C,19.2125,50.0000,30.7875



========== DUAIS (preços-sombra) ==========
Demanda por tipo (US$/m): {'p': '0.076923', 'm': '0.114286', 'g': '0.133333', 'e': '0.246154'}
Capacidade por máquina (US$/h): {'A': '-0.769231', 'B': '0.000000', 'C': '0.000000'}


# Comentários

- **Custo ótimo**: **US$ 3.960,07**.

- **Alocação ótima (horas → produção)**  
  - **pequena (p)**: Maq **B** = 18,4615 h → **12.000 m**  
  - **mediana (m)**: Maq **C** = 8,5714 h → **6.000 m**  
  - **grande (g)**: Maq **C** = 8,3333 h → **5.000 m**  
  - **extra (e)**: Maq **A** = 50,0 h → 6.250 m; Maq **C** = 2,3077 h → 750 m (fecha **7.000 m**)

- **Uso de capacidade (h)**  
  - **A**: 50,0 / 50,0 (**ativa**)  
  - **B**: 18,46 / 50,0 (folga 31,54)  
  - **C**: 19,21 / 50,0 (folga 30,79)

- **Por que essa alocação é intuitiva (custo por metro $c_m/r_{t,m}$)**  
  | Tipo | A (\$/m) | B (\$/m) | C (\$/m) | **Mais barato** |
  |---|---:|---:|---:|---|
  | p | 0,0857 | **0,0769** | 0,0941 | **B** |
  | m | 0,1200 | 0,1250 | **0,1143** | **C** |
  | g | 0,1500 | 0,1429 | **0,1333** | **C** |
  | e | **0,2400** | 0,2500 | 0,2462 | **A** |
  → O modelo escolhe, para cada tipo, a **máquina de menor custo por metro**, limitado pela **capacidade de A** (50 h).

- **Restrições ativas**  
  - Todas as **demandas** são atendidas **exatamente** (sem excesso).  
  - **Capacidade da máquina A** é **limitante**; B e C têm folga.

- **Preços-sombra (duais)**  
  - Demandas (US\$/m): p **0,076923**, m **0,114286**, g **0,133333**, e **0,246154**  
    → Aumentar a demanda de um tipo em **1 m** eleva o custo ótimo em **Pi** US$.  
  - Capacidade de A (US\$/h): **−0,769231**  
    → **+1 h** de capacidade em A **reduz** o custo em ≈ **US$ 0,769** (B e C têm Pi = 0).

- **Análise “e se?” rápida**  
  - Os **750 m** de “extra” hoje feitos na **C** poderiam migrar para **A** se houvesse **+6 h** em A (750/125).  
  - Economia esperada ≈ $6 \times 0{,}769 \approx $ **US$ 4,62** — pequena frente ao custo total, logo o gargalo em A não é muito “caro”.

- **Diagnóstico geral**  
  A solução é coerente com um **problema de alocação de custo mínimo**: usar a máquina **mais barata por metro** para cada tipo, respeitando limites de horas; apenas a **A** satura e força uma pequena parcela das vigas **extra** a ser feita na **C**.



In [3]:
# quick model stats
print(f"#vars={m.NumVars}, #constr={m.NumConstrs}, #nz={m.NumNZs}")
print("Sense:", "min" if m.ModelSense == GRB.MINIMIZE else "max")
print("Status:", m.Status)                    # 2=optimal, etc.
if m.Status == GRB.OPTIMAL:
    print("Obj =", m.ObjVal)

# list variables (name, bounds, obj coeff)
for v in m.getVars():
    print(v.VarName, v.LB, v.UB, v.Obj)

# list constraints: LHS expression and RHS
for c in m.getConstrs():
    print(c.ConstrName, ":", m.getRow(c), "<=/>=/=" , c.RHS)  # sense shown by row vs RHS
    # or, if you want the sense explicitly:
    # print(c.ConstrName, c.Sense, m.getRow(c), "=", c.RHS)

# after optimize(): print variable values, duals and slacks
m.printAttr("X")       # values of variables
m.printAttr("Pi")      # duals (linear constraints)
m.printAttr("Slack")   # primal slacks
# also useful:
m.printAttr("RC")      # reduced costs (vars)
m.printAttr("Obj")     # objective coeffs (vars)

# model quality / stats
m.printStats()
m.printQuality()

# dump model/solution to files for inspection
m.write("model.lp")    # human-readable model
m.write("solution.sol")


#vars=12, #constr=7, #nz=24
Sense: min
Status: 2
Obj = 3960.07326007326
h[p,A] 0.0 inf 30.0
h[p,B] 0.0 inf 50.0
h[p,C] 0.0 inf 80.0
h[m,A] 0.0 inf 30.0
h[m,B] 0.0 inf 50.0
h[m,C] 0.0 inf 80.0
h[g,A] 0.0 inf 30.0
h[g,B] 0.0 inf 50.0
h[g,C] 0.0 inf 80.0
h[e,A] 0.0 inf 30.0
h[e,B] 0.0 inf 50.0
h[e,C] 0.0 inf 80.0
dem_p : 350.0 h[p,A] + 650.0 h[p,B] + 850.0 h[p,C] <=/>=/= 12000.0
dem_m : 250.0 h[m,A] + 400.0 h[m,B] + 700.0 h[m,C] <=/>=/= 6000.0
dem_g : 200.0 h[g,A] + 350.0 h[g,B] + 600.0 h[g,C] <=/>=/= 5000.0
dem_e : 125.0 h[e,A] + 200.0 h[e,B] + 325.0 h[e,C] <=/>=/= 7000.0
cap_A : h[p,A] + h[m,A] + h[g,A] + h[e,A] <=/>=/= 50.0
cap_B : h[p,B] + h[m,B] + h[g,B] + h[e,B] <=/>=/= 50.0
cap_C : h[p,C] + h[m,C] + h[g,C] + h[e,C] <=/>=/= 50.0

    Variable            X 
-------------------------
      h[p,B]      18.4615 
      h[m,C]      8.57143 
      h[g,C]      8.33333 
      h[e,A]           50 
      h[e,C]      2.30769 

  Constraint           Pi 
-------------------------
       dem_p   

# Questão 5

**Objetivo.** Maximizar o **lucro diário** produzindo dois tipos de produto:
- Produto 1: requer $ \tfrac{1}{4} $ h de **montagem**, $ \tfrac{1}{8} $ h de **teste** e tem custo de matérias-primas **1,2**; preço de venda **9**.
- Produto 2: requer $ \tfrac{1}{3} $ h de **montagem**, $ \tfrac{1}{3} $ h de **teste** e tem custo de matérias-primas **0,9**; preço de venda **8**.

**Capacidades diárias:** montagem $\le 90$ h; teste $\le 80$ h.

**Variáveis.** $x_1, x_2 \ge 0$: unidades diárias dos produtos 1 e 2.

**Lucros unitários.** $\pi_1 = 9-1.2=7.8$, $\pi_2 = 8-0.9=7.1$.

**Modelo.**
$$
\max\; 7.8\,x_1 + 7.1\,x_2
\quad \text{s.a.} \quad
\frac{1}{4}x_1 + \frac{1}{3}x_2 \le 90,\;
\frac{1}{8}x_1 + \frac{1}{3}x_2 \le 80,\;
x_1,x_2 \ge 0.
$$

A seguir, implementamos e resolvemos o modelo no **Gurobi** e exibimos os resultados
(lucro ótimo, plano de produção, uso de recursos, folgas e preços-sombra).


In [8]:
# -----------------------------
# Parâmetros
# -----------------------------
products = ["p1", "p2"]

# lucro unitário (preço - matéria-prima)
# profit = {"p1": 7.8, "p2": 7.1}
profit = {"p1": 7.8, "p2": 10.4}

# requisitos de tempo (h/unidade)
assembly = {"p1": 1/4, "p2": 1/3}  # montagem
test     = {"p1": 1/8, "p2": 1/3}  # teste

# capacidades diárias (h)
cap = {"assembly": 90.0, "test": 80.0}

# -----------------------------
# Modelo
# -----------------------------
m5 = Model("q5_production")
m5.Params.OutputFlag = 1  # exibir log

# Variáveis: produção diária
x = m5.addVars(products, name="x", lb=0.0, vtype=GRB.CONTINUOUS)

# Objetivo: maximizar lucro total
m5.setObjective(quicksum(profit[p] * x[p] for p in products), GRB.MAXIMIZE)

# Restrições de capacidade
constr_assembly = m5.addConstr(quicksum(assembly[p] * x[p] for p in products) <= cap["assembly"],
                              name="cap_assembly")
constr_test     = m5.addConstr(quicksum(test[p]     * x[p] for p in products) <= cap["test"],
                              name="cap_test")

# Opcional: escrever arquivo LP
m5.write("q5_production.lp")

# Otimiza
m5.optimize()

# -----------------------------
# Relatórios / Saída formatada
# -----------------------------
if m5.SolCount > 0:
    # Plano de produção
    plan = pd.Series({p: x[p].X for p in products}, name="Unidades")
    plan.index = ["Produto 1 (x1)", "Produto 2 (x2)"]

    # Uso de recursos
    use = {
        "Montagem (h)": sum(assembly[p] * x[p].X for p in products),
        "Teste (h)":    sum(test[p]     * x[p].X for p in products),
    }
    use_df = pd.DataFrame({
        "Uso (h)": pd.Series(use),
        "Capacidade (h)": pd.Series({"Montagem (h)": cap["assembly"], "Teste (h)": cap["test"]})
    })
    use_df["Folga (h)"] = use_df["Capacidade (h)"] - use_df["Uso (h)"]

    # Lucro ótimo
    z_opt = m5.ObjVal

    # Impressão
    print("\n========== LUCRO ÓTIMO ==========")
    print(f"z* = {z_opt:,.2f} (unidades monetárias)")

    print("\n========== PLANO DE PRODUÇÃO (x1, x2) ==========")
    display(plan.to_frame())

    print("\n========== USO DE RECURSOS ==========")
    display(use_df.style.format("{:.4f}"))

    # Preços-sombra (duais) das capacidades
    try:
        pi_assembly = constr_assembly.Pi
        pi_test     = constr_test.Pi
        print("\n========== PREÇOS-SOMBRA (duais) ==========")
        print(f"Montagem: {pi_assembly:.6f}  |  Teste: {pi_test:.6f}")
        # Reduzidos das variáveis
        print("\n========== CUSTOS REDUZIDOS (variáveis) ==========")
        for p in products:
            print(f"{x[p].VarName}: RC = {x[p].RC:.6f}")
    except AttributeError:
        pass
else:
    print("Modelo inviável ou sem solução.")


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i9-14900HX, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0xcea89608
Coefficient statistics:
  Matrix range     [1e-01, 3e-01]
  Objective range  [8e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 9e+01]
Presolve time: 0.01s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.6000000e+31   2.833333e+30   2.600000e+01      0s
       2    2.8080000e+03   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.808000000e+03

========== LUCRO ÓTIMO ==========
z* = 2,808.00 (unidades monetárias)

========== PLANO DE PRODUÇÃO (x1, x2) ==========


,Unidades
Produto 1 (x1),80.0
Produto 2 (x2),210.0



========== USO DE RECURSOS ==========


,Uso (h),Capacidade (h),Folga (h)
Montagem (h),90.0000,90.0000,0.0000
Teste (h),80.0000,80.0000,0.0000



========== PREÇOS-SOMBRA (duais) ==========
Montagem: 31.200000  |  Teste: 0.000000

========== CUSTOS REDUZIDOS (variáveis) ==========
x[p1]: RC = 0.000000
x[p2]: RC = 0.000000


# Comentários

- **Lucro ótimo**: $ z^\star = \mathbf{2{,}808.00} $.

- **Plano ótimo**: $ x_1^\star = \mathbf{360} $ unidades do produto 1; $ x_2^\star = \mathbf{0} $ do produto 2.

- **Uso de recursos**
  - **Montagem**: $90/90$ h (**ativa**, folga $0$ h).
  - **Teste**: $45/80$ h (folga $35$ h).  
  → **Gargalo**: montagem.

- **Por que produzimos só o produto 1?**  
  Compare o **lucro por hora do recurso limitante (montagem)**:
  - $p_1$: lucro unit. $7{,}8$, montagem $0{,}25$ h → $7{,}8 / 0{,}25 = \mathbf{31{,}2}$ por h.  
  - $p_2$: lucro unit. $7{,}1$, montagem $1/3$ h → $7{,}1 / (1/3) = \mathbf{21{,}3}$ por h.  
  Como **31,2 > 21,3**, vale alocar toda a montagem a $p_1$ até saturar a capacidade (resulta nos 360 unid.).

- **Preços-sombra (duais)**
  - **Montagem**: $\pi_{\text{mont}} = \mathbf{31{,}2}$ (UM/h).  
    → Ganho marginal: **+1 h** de montagem aumenta o lucro ótimo em **≈ 31,2**.  
    (Consistente com “+4 unidades de $p_1$” por hora extra: $4 \times 7{,}8 = 31{,}2$.)
  - **Teste**: $\pi_{\text{teste}} = 0$ (recurso folgado; horas extras de teste **não** alteram o ótimo).

- **Custos reduzidos (variáveis)**
  - $RC(x_1)=0$ (básica no ótimo).  
  - $RC(x_2) = \mathbf{-3{,}3}$. Em maximização, isso significa que o **lucro unitário de $p_2$** teria de **aumentar em 3,3** (de 7,1 para **10,4**) para que fosse atrativo produzir $p_2$ no ótimo atual.  
    Equivalentemente: para competir pelo gargalo (montagem), $p_2$ precisaria ter **lucro por hora de montagem** pelo menos $31{,}2$, o que implica lucro unitário $\ge 31{,}2 \times \tfrac{1}{3} = 10{,}4$.

- **“E se…?”**
  - **+1 h de montagem** → $z^\star$ sobe ≈ **31,2**; plano tende a aumentar $x_1$ (recurso teste ainda folgado).  
  - **+1 h de teste** → nenhum efeito (dual $=0$).  
  - Mesmo com muita montagem, $p_1$ também domina em **teste** (lucro/h de teste: $p_1=62{,}4$ vs $p_2=21{,}3$), então só passamos a produzir $p_2$ se **seu lucro unitário melhorar** (ou se seus consumos de recurso diminuírem).
